In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

import ripser
import persim

In [2]:
########## read the data ##########

processes = [f"run_{i:02d}_decayed_1" for i in range(1,12)]
kappa_3 = [0.2, 0.4, 0.6, 0.8, 0.9, 1, 1.1, 1.2, 1.4, 1.6, 1.8]
# Xsection_NoDecay = [1.23322e-01, 1.016366e-01, 8.33476424e-02, 6.81961e-02, 6.2211286e-02, 5.648398e-02, 5.187586e-02, 4.801255e-02, 4.246736e-02, 4.038406e-02, 4.1308219e-02]     ### hhvv unit: fb
Xsection_NoDecay = [1.068573e-2, 8.89267e-3, 7.29573e-3, 5.892737e-3, 5.2836183e-3, 4.68848e-3, 4.1895e-3, 3.7137245e-3, 2.9218193e-3, 2.3228e-3, 1.9414667e-3]     ### hhmumu unit: fb
BR_hbb = 0.5824
Xsection = []
for i in range(11):
    Xsection.append(Xsection_NoDecay[i]*BR_hbb*BR_hbb)

Luminosity = 1000   ### unit: fb^-1
simulation_num = 100000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Eta", "VLCjetR10N2.Phi", "VLCjetR10N2.Mass", "MissingET.MET", "VLCjetR10N2.BTag", "EFlow.Eta", "EFlow.Phi", "EFlow.ET"]

##### set data path #####

event_path = []
for type in processes:
    event_path.append("/data/mucollider/two_boosted/scan_kappa3_hhmumu/Events/" + type + "/delphes_events.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print(process, "Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

run_01_decayed_1 Time: 4.3817 (s)
run_02_decayed_1 Time: 7.2780 (s)
run_03_decayed_1 Time:10.2090 (s)
run_04_decayed_1 Time:13.1532 (s)
run_05_decayed_1 Time:16.1763 (s)
run_06_decayed_1 Time:18.9620 (s)
run_07_decayed_1 Time:22.0763 (s)
run_08_decayed_1 Time:24.8648 (s)
run_09_decayed_1 Time:28.0672 (s)
run_10_decayed_1 Time:30.8896 (s)
run_11_decayed_1 Time:33.6665 (s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### define X_HH (defined in 2207.09602v2) #####

def calculate_X_HH(event):
    jet_mass1 = event[features.index("VLCjetR10N2.Mass")][0]
    jet_mass2 = event[features.index("VLCjetR10N2.Mass")][1]
    diff1 = np.abs(jet_mass1 - 124)
    diff2 = np.abs(jet_mass2 - 124)
    if diff1<diff2:
        m1 = jet_mass1
        m2 = jet_mass2
    else:
        m1 = jet_mass2
        m2 = jet_mass1
    return np.sqrt(((m1-124)/(0.1*m1+0.00001))**2 + ((m2-115)/(0.1*m2+0.00001))**2)

##### calculate M_JJ #####

def calculate_M_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    M_JJ = np.sqrt(p[0]**2 - p[1]**2 - p[2]**2 - p[3]**2)
    return M_JJ

##### calculate pT_JJ #####

def calculate_pT_JJ(event):
    p = [0,0,0,0]    ### four momentum
    for i in range(2):    ### leading jet and sub-leading jet
        pt = event[features.index("VLCjetR10N2.PT")][i]
        eta = event[features.index("VLCjetR10N2.Eta")][i]
        phi = event[features.index("VLCjetR10N2.Phi")][i]
        mass = event[features.index("VLCjetR10N2.Mass")][i]
        p[1] = p[1] + pt*np.cos(phi)    ### px
        p[2] = p[2] + pt*np.sin(phi)    ### py
        p[3] = p[3] + pt*np.sinh(eta)    ### pz
        p[0] = p[0] + np.sqrt( mass**2 + (pt*np.cos(phi))**2 + (pt*np.sin(phi))**2 + (pt*np.sinh(eta))**2 )    ### energy
    pT_JJ = np.sqrt(p[1]**2 + p[2]**2)
    return pT_JJ

##### calculate \Delta_\Eta #####

def calculate_dEta_JJ(event):
    return np.abs(event[features.index("VLCjetR10N2.Eta")][0] - event[features.index("VLCjetR10N2.Eta")][1])

##### calculate TDA #####

def calculate_TDA(event):
#     neutrino_ID = [12, 14, 16, -12, -14, -16]
#     where_jet_particle = np.where(event[features.index("Particle.Status")]==1)[0]
#     where_neutrino = np.where(np.isin(event[features.index("Particle.PID")], neutrino_ID))[0]
#     where_jet_particle = np.delete(where_jet_particle, np.isin(where_jet_particle, where_neutrino))

    particle_ET = event[features.index("EFlow.ET")]
    particle_ETcut = 0
    where_ET_larger = np.where(particle_ET>particle_ETcut)
    particle_ET = event[features.index("EFlow.ET")][where_ET_larger]
    
    center_eta = (event[features.index("VLCjetR10N2.Eta")][0]+event[features.index("VLCjetR10N2.Eta")][1])/2    ### centerize the event
    center_phi = (event[features.index("VLCjetR10N2.Phi")][0]+event[features.index("VLCjetR10N2.Phi")][1])/2
    particle_eta = event[features.index("EFlow.Eta")][where_ET_larger]-center_eta
    particle_phi = event[features.index("EFlow.Phi")][where_ET_larger]-center_phi
    
    where_larger_pi = np.where(particle_phi>np.pi)
    particle_phi[where_larger_pi] = -(2*np.pi - particle_phi[where_larger_pi])    ### check if jet particle split
    where_smaller_mpi = np.where(particle_phi<-np.pi)
    particle_phi[where_smaller_mpi] = 2*np.pi + particle_phi[where_smaller_mpi]    ### check if jet particle split

    P = np.array([particle_eta, particle_phi]).T   ### calculate TDA
    diagrams = ripser.ripser(P)['dgms']
    
    return diagrams

##### calculate TDA sum of lifetime #####

def TDA_sum(diagrams):
    sum_l0 = np.sum(diagrams[0][:-1,1] - diagrams[0][:-1,0])
    sum_l1 = np.sum(diagrams[1][:,1] - diagrams[1][:,0])
    return sum_l0, sum_l1

##### calculate TDA mean of lifetime #####

def TDA_mean(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    if np.sum(l1)==0:
        l1 = np.concatenate([l1,[0]])
    return np.mean(l0), np.mean(l1)

##### calculate TDA entropy #####

def TDA_entropy(diagrams):
    l0 = diagrams[0][:-1,1]-diagrams[0][:-1,0]
    L0 = np.sum(l0)
    S0 = (l0/L0)*np.log2((l0/L0))
    l1 = diagrams[1][:,1]-diagrams[1][:,0]
    L1 = np.sum(l1)
    S1 = (l1/L1)*np.log2((l1/L1))
    return -np.sum(S0), -np.sum(S1)

##### calculate TDA Lifetime*Birthtime #####

def TDA_LB(diagrams):
    return np.sum(diagrams[1][:,0]*(diagrams[1][:,1]-diagrams[1][:,0]))

In [4]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 800   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 100000 run_01_decayed_1 events. Corresponding cross section: 0.0036244901941248 (fb)
There are 100000 run_02_decayed_1 events. Corresponding cross section: 0.0030163026030592005 (fb)
There are 100000 run_03_decayed_1 events. Corresponding cross section: 0.0024746369077248004 (fb)
There are 100000 run_04_decayed_1 events. Corresponding cross section: 0.00199875604877312 (fb)
There are 100000 run_05_decayed_1 events. Corresponding cross section: 0.001792149223108608 (fb)
There are 100000 run_06_decayed_1 events. Corresponding cross section: 0.0015902844059648004 (fb)
There are 100000 run_07_decayed_1 events. Corresponding cross section: 0.00142103549952 (fb)
There are 100000 run_08_decayed_1 events. Corresponding cross section: 0.0012596573218611202 (fb)
There are 100000 run_09_decayed_1 events. Corresponding cross section: 0.000991051187130368 (fb)
There are 100000 run_10_decayed_1 events. Corresponding cross section: 0.000787869974528 (fb)
There are 1000

In [5]:
##### calculate the high level features #####

num_of_events = []

m_J_leading = []
m_J_subleading = []
pt_J_leading = []
pt_J_subleading = []
missET = []
X_HH = []
M_JJ = []
pT_JJ = []
dEta_JJ = []
BTag_leading = []
BTag_subleading = []

for process in processes:
    num_of_events.append(len(events[processes.index(process)]))
    for i in range(len(events[processes.index(process)])):
        m_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][0])
        m_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.Mass")][1])
        pt_J_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][0])
        pt_J_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.PT")][1])
        missET.append(events[processes.index(process)][i][features.index("MissingET.MET")][0])
        X_HH.append(calculate_X_HH(events[processes.index(process)][i]))
        M_JJ.append(calculate_M_JJ(events[processes.index(process)][i]))
        pT_JJ.append(calculate_pT_JJ(events[processes.index(process)][i]))
        dEta_JJ.append(calculate_dEta_JJ(events[processes.index(process)][i]))
        BTag_leading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][0])
        BTag_subleading.append(events[processes.index(process)][i][features.index("VLCjetR10N2.BTag")][1])
    print("Time:{:^8.4f}(s)".format(time.time()-t1))
    
print("Number of events:", num_of_events)

Time:57.0071 (s)
Time:60.6294 (s)
Time:64.2488 (s)
Time:67.8542 (s)
Time:71.4399 (s)
Time:74.9859 (s)
Time:78.5035 (s)
Time:82.0436 (s)
Time:85.4442 (s)
Time:88.7299 (s)
Time:91.9464 (s)
Number of events: [30469, 30020, 29858, 29497, 29190, 28845, 28653, 28343, 27442, 26613, 25744]


In [6]:
##### calculate TDA high level features #####

H_0 = []    ### sum of 0'th order lifetime 
H_1 = []    ### sum of 1'th order lifetime 
S_0 = []    ### sum of 0'th order entropy
S_1 = []    ### sum of 1'th order entropy
LB_1 = []   ### sum of 1'th order Lifetime*Birthtime

for process in processes:
    for i in range(len(events[processes.index(process)])):
        if (i%20000)==19999:
            print("check point")
        TDA_tmp = calculate_TDA(events[processes.index(process)][i])
        H_0.append(TDA_sum(TDA_tmp)[0])        
        H_1.append(TDA_sum(TDA_tmp)[1])
        S_0.append(TDA_entropy(TDA_tmp)[0])   
        S_1.append(TDA_entropy(TDA_tmp)[1])
        LB_1.append(TDA_LB(TDA_tmp))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

check point
Time:196.3876(s)
check point
Time:300.5226(s)
check point
Time:402.8331(s)
check point
Time:504.0173(s)
check point
Time:604.0192(s)
check point
Time:702.8518(s)
check point
Time:802.3027(s)
check point
Time:899.6263(s)
check point
Time:994.3189(s)
check point
Time:1086.0845(s)
check point
Time:1175.5555(s)


In [7]:
##### make target #####

target_kappa_3 = np.repeat(kappa_3, num_of_events)

In [8]:
##### output test data #####

d = {'m_J_leading':m_J_leading, 'm_J_subleading':m_J_subleading, 'pt_J_leading':pt_J_leading, 'pt_J_subleading':pt_J_subleading, 'missET':missET, 'X_HH':X_HH, 'M_JJ':M_JJ, 'pT_JJ':pT_JJ, 'dEta_JJ':dEta_JJ, 'BTag_leading':BTag_leading, 'BTag_subleading':BTag_subleading, 'H_0':H_0, 'H_1':H_1, 'S_0':S_0, 'S_1':S_1, 'LB_1':LB_1, 'target_kappa_3':target_kappa_3}
df = pd.DataFrame(data=d)
df

,m_J_leading,m_J_subleading,pt_J_leading,pt_J_subleading,missET,X_HH,M_JJ,pT_JJ,dEta_JJ,BTag_leading,BTag_subleading,H_0,H_1,S_0,S_1,LB_1,target_kappa_3
0,122.008400,121.862541,354.149841,322.584015,95.513046,0.586318,925.812314,95.590073,1.564034,2,5,8.882102,0.020377,4.403374,1.475059,0.001830,0.2
1,111.333755,87.811241,395.070892,309.313324,94.823372,3.298667,1515.317963,89.580633,2.797453,7,7,19.350427,0.337637,5.798205,3.570794,0.071749,0.2
2,119.561256,102.655106,296.937042,286.854492,100.913757,1.258561,1179.842501,101.602851,2.620732,7,6,12.076723,0.209584,5.122494,2.716021,0.057023,0.2
3,129.578857,101.582367,245.633438,213.896774,23.417082,1.389258,587.135456,32.756372,1.109929,6,1,12.654116,0.339332,5.496070,2.823118,0.092533,0.2
4,124.719978,123.747406,347.563293,205.267395,212.948456,0.779611,651.463951,215.566787,1.128525,3,7,9.003201,0.127320,5.131661,2.915248,0.027380,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314669,119.825928,95.789268,694.910645,524.908569,148.331635,2.035546,1276.164349,170.056400,0.571788,7,6,8.507346,0.236113,5.561185,2.977236,0.042515,1.8
314670,122.966698,100.951477,404.144562,260.792877,138.834030,1.394145,1094.477090,160.929838,2.167971,7,4,9.824695,0.075930,4.786147,2.076934,0.009450,1.8
314671,81.005470,119.195602,256.886444,217.049423,139.612427,4.215879,833.542957,139.592433,2.238365,4,6,9.742358,0.080000,4.970561,2.447645,0.010737,1.8
314672,125.410675,113.019012,319.729523,203.143188,185.019058,0.208268,776.507578,184.650121,1.882643,6,6,14.528634,0.284371,5.974746,2.321061,0.077109,1.8


In [9]:
h5f = h5py.File('/data/mucollider/two_boosted/scan_kappa3_hhmumu_HL.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")

已寫入 Dataset: m_J_leading
已寫入 Dataset: m_J_subleading
已寫入 Dataset: pt_J_leading
已寫入 Dataset: pt_J_subleading
已寫入 Dataset: missET
已寫入 Dataset: X_HH
已寫入 Dataset: M_JJ
已寫入 Dataset: pT_JJ
已寫入 Dataset: dEta_JJ
已寫入 Dataset: BTag_leading
已寫入 Dataset: BTag_subleading
已寫入 Dataset: H_0
已寫入 Dataset: H_1
已寫入 Dataset: S_0
已寫入 Dataset: S_1
已寫入 Dataset: LB_1
已寫入 Dataset: target_kappa_3


In [10]:
print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:1177.8285(s)


In [11]:
h5f.close()